In [ ]:
from pathlib import Path

import pandas as pd

In [ ]:
# --------------------------------------------------
# Directories
# --------------------------------------------------
DIR_DATA = Path("data")
DIR_METADATA = DIR_DATA / "0_metadata"
DIR_PROCESSED = DIR_DATA / "2_processed"
DIR_RESULTS = DIR_DATA / "3_results"
DIR_EVAL = DIR_DATA / "4_evaluation"

DIR_ICEYE = DIR_PROCESSED / "1_ICEYE"
DIR_NDVI = DIR_PROCESSED / "2_Sentinel2"
DIR_DEPMAP = DIR_PROCESSED / "3_Terrain" / "depmap"
DIR_ZSCORE = DIR_PROCESSED / "4_ICEYE_Zscore"
DIR_GRADCAM_CLS = DIR_RESULTS / "siamese-gradcam-classification" / "patches-gradcam"
DIR_GRADCAM_REG = DIR_RESULTS / "siamese-gradcam-regression" / "patches-gradcam"

# --------------------------------------------------
# Files
# --------------------------------------------------
FILEPATH_PAIR_MANIFEST = DIR_METADATA / "pair-manifest.csv"
FILEPATH_EVAL_SUMMARY = DIR_EVAL / "correlation_summary.csv"
FILEPATH_PATCH_CENTERS = DIR_METADATA / "patch-centers.csv"


TARGET_PAIRS = ["WE01", ]

CORRELATION_METHOD = "spearman"

In [ ]:
def find_gradcam_file(directory, pair_id, patch_id):
    pattern = f"gradcam_{pair_id}_{patch_id}_*.tif"
    matches = sorted(directory.glob(pattern))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"{pattern}: expected 1 file, found {len(matches)}"
        )
    return matches[0]

In [ ]:
# --------------------------------------------------
# Read manifest
# --------------------------------------------------
manifest = pd.read_csv(FILEPATH_PAIR_MANIFEST)

# Only WE01 for now
manifest = manifest[manifest["pair_id"].isin(TARGET_PAIRS)]

rows = []
for _, pair in manifest.iterrows():
    pair_id = pair["pair_id"]
    year = pair["sar_target"][:4]

    sar_target_dir = DIR_ICEYE / pair["sar_target"]
    sar_base_dir = DIR_ICEYE / pair["sar_base"]
    ndvi_dir = (
        DIR_NDVI
        / f"patches_{year}"
        / pair["ndvi_target"]
    )
    depmap_dir = (
        DIR_DEPMAP
        / f"patches_{year}"
    )
    zscore_dir = (
        DIR_ZSCORE
        / pair["sar_target"]
    )
    gradcam_cls_dir = DIR_GRADCAM_CLS
    gradcam_reg_dir = DIR_GRADCAM_REG

    # ----------------------------------------------
    # Enumerate ICEYE test patches
    # ----------------------------------------------
    for sar_target_file in sorted(sar_target_dir.glob("*.tif")):

        stem = sar_target_file.stem

        # Example:
        # 4155938_EB24CC0120

        image_id, patch_id = stem.split("_", 1)

        # Test patches begin with E
        if not patch_id.startswith("E"):
            continue

        sar_base_file = (
            sar_base_dir
            / f"{pair["sar_base"].split("_")[-1]}_{patch_id}.tif"
        )
        ndvi_file = (
            ndvi_dir
            / f"{pair['ndvi_target']}_{patch_id}_NDVI.tif"
        )
        depmap_file = (
            depmap_dir
            / f"depmap_{patch_id}.tif"
        )
        zscore_file = (
            zscore_dir
            / f"z_{sar_target_file.name}"
        )
        gradcam_cls_file = find_gradcam_file(
            gradcam_cls_dir,
            pair_id,
            patch_id,
        )
        gradcam_reg_file = find_gradcam_file(
            gradcam_reg_dir,
            pair_id,
            patch_id,
        )
        rows.append(
            {
                "pair_id": pair_id,
                "patch_id": patch_id,
                "sar_base": sar_base_file,
                "sar_target": sar_target_file,
                "zscore": zscore_file,
                "ndvi": ndvi_file,
                "depmap": depmap_file,
                "gradcam_cls": gradcam_cls_file,
                "gradcam_reg": gradcam_reg_file,
            }
        )

groups_patch_paths = pd.DataFrame(rows)

print(groups_patch_paths.head())
print(f"\n{len(groups_patch_paths)} test patches found.")

In [ ]:
import numpy as np
import pandas as pd
import rasterio
from scipy.stats import pearsonr, spearmanr

In [ ]:
def compute_correlation(image1, image2, method="spearman"):
    """
    Compute the correlation between two raster images.

    Parameters
    ----------
    image1 : np.ndarray
        First raster (e.g., Grad-CAM).
    image2 : np.ndarray
        Second raster (e.g., Z-score, depmap, NDVI).
    method : str, optional
        "spearman" (default) or "pearson".

    Returns
    -------
    float
        Correlation coefficient.
    """
    x = image1.ravel()
    y = image2.ravel()

    # Remove NaN values if present
    valid = np.isfinite(x) & np.isfinite(y)
    x = x[valid]
    y = y[valid]

    if len(x) < 2:
        return np.nan

    method = method.lower()

    if method == "spearman":
        corr, _ = spearmanr(x, y)
    elif method == "pearson":
        corr, _ = pearsonr(x, y)
    else:
        raise ValueError(
            "method must be 'spearman' or 'pearson'"
        )

    return corr

In [ ]:
results = []
for _, row in groups_patch_paths.iterrows():
    # ------------------------------
    # Read rasters
    # ------------------------------
    with rasterio.open(row["zscore"]) as src:
        zscore = src.read(1)

    with rasterio.open(row["depmap"]) as src:
        depmap = src.read(1)

    with rasterio.open(row["ndvi"]) as src:
        ndvi = src.read(1)

    with rasterio.open(row["gradcam_cls"]) as src:
        gradcam_cls = src.read(1)

    with rasterio.open(row["gradcam_reg"]) as src:
        gradcam_reg = src.read(1)

    # ------------------------------
    # Compute correlations
    # ------------------------------
    results.append({
        "pair_id": row["pair_id"],
        "patch_id": row["patch_id"],
        "cls_prob": float(
            row["gradcam_cls"].stem.split("_prob")[-1]
        ),
        "reg_pred": float(
            row["gradcam_reg"].stem.split("_pred")[-1]
        ),
        "corr_cls_zscore": compute_correlation(
            gradcam_cls, zscore, method=CORRELATION_METHOD
        ),
        "corr_cls_depmap": compute_correlation(
            gradcam_cls, depmap, method=CORRELATION_METHOD
        ),
        "corr_cls_ndvi": compute_correlation(
            gradcam_cls, ndvi, method=CORRELATION_METHOD
        ),
        "corr_reg_zscore": compute_correlation(
            gradcam_reg, zscore, method=CORRELATION_METHOD
        ),
        "corr_reg_depmap": compute_correlation(
            gradcam_reg, depmap, method=CORRELATION_METHOD
        ),
        "corr_reg_ndvi": compute_correlation(
            gradcam_reg, ndvi, method=CORRELATION_METHOD
        ),
        })

eval_summary = pd.DataFrame(results)
eval_summary.head()

In [ ]:
eval_summary.to_csv(
    FILEPATH_EVAL_SUMMARY,
    index=False,
)

In [ ]:
from scipy.stats import spearmanr
import matplotlib.pyplot as plt

In [ ]:
# ---------------------------------------
# Prepare data
# ---------------------------------------
plot_data = eval_summary[
    [
        "patch_id",
        "cls_prob",
        "reg_pred",
    ]
].dropna().copy()


# ---------------------------------------
# Spearman correlation
# ---------------------------------------
rho, p_value = spearmanr(
    plot_data["cls_prob"],
    plot_data["reg_pred"],
)

print(f"Spearman rho: {rho:.3f}")
print(f"p-value     : {p_value:.4f}")


# ---------------------------------------
# Plot
# ---------------------------------------
fig, ax = plt.subplots(
    figsize=(7, 6)
)

ax.scatter(
    plot_data["cls_prob"],
    plot_data["reg_pred"],
    color="gray",
    s=50,
    alpha=0.7,
)


# ---------------------------------------
# Patch IDs
# ---------------------------------------
for _, row in plot_data.iterrows():
    ax.text(
        row["cls_prob"],
        row["reg_pred"],
        row["patch_id"],
        fontsize=7,
    )


# ---------------------------------------
# Labels
# ---------------------------------------
ax.set_xlabel(
    "Classification probability"
)

ax.set_ylabel(
    "Regression prediction"
)

ax.set_xlim(
    0,
    1,
)

ax.grid(
    alpha=0.3,
)

ax.set_title(
    f"Classification vs. Regression Output\n"
    f"Spearman $\\rho$ = {rho:.3f}, "
    f"$p$ = {p_value:.3g}"
)

plt.tight_layout()


# ---------------------------------------
# Save
# ---------------------------------------
output_path = (
    DIR_EVAL /
    "fig_siamese_classification_vs_regression_wet.png"
)

fig.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight",
)

print(f"Saved: {output_path}")


# ---------------------------------------
# Display
# ---------------------------------------
plt.show()
plt.close(fig)

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# ---------------------------------------
# Convert wide table to long table
# ---------------------------------------
plot_df = eval_summary.melt(
    id_vars=["pair_id", "patch_id"],
    value_vars=[
        "corr_cls_zscore",
        "corr_cls_depmap",
        "corr_cls_ndvi",
        "corr_reg_zscore",
        "corr_reg_depmap",
        "corr_reg_ndvi",
    ],
    var_name="metric",
    value_name="correlation",
)

# Remove NaN correlations
plot_df = plot_df.dropna(subset=["correlation"])

# ---------------------------------------
# Split metric into model/reference
# ---------------------------------------
plot_df["model"] = plot_df["metric"].apply(
    lambda x: "Classification" if "_cls_" in x else "Regression"
)

plot_df["reference"] = (
    plot_df["metric"]
    .str.replace("corr_cls_", "", regex=False)
    .str.replace("corr_reg_", "", regex=False)
)

plot_df["reference"] = plot_df["reference"].replace({
    "zscore": "Z-score",
    "depmap": "Depression map",
    "ndvi": "NDVI",
})

In [ ]:
# ---------------------------------------
# Figure 1 settings
# ---------------------------------------
THRESHOLD_CLS = 0.5
THRESHOLD_REG = 10.0


# ---------------------------------------
# Prediction quality
# ---------------------------------------
quality = eval_summary[
    [
        "pair_id",
        "patch_id",
        "cls_prob",
        "reg_pred",
    ]
].copy()

# All samples in this analysis are rain samples.
# Model-specific thresholds:
quality["cls_good"] = (
    quality["cls_prob"] >= THRESHOLD_CLS
)

quality["reg_good"] = (
    quality["reg_pred"] >= THRESHOLD_REG
)


# ---------------------------------------
# Add quality information to plot_df
# ---------------------------------------
plot_df_fig1 = plot_df.merge(
    quality[
        [
            "pair_id",
            "patch_id",
            "cls_good",
            "reg_good",
        ]
    ],
    on=[
        "pair_id",
        "patch_id",
    ],
    how="left",
)


# ---------------------------------------
# Figure 1
# ---------------------------------------
fig, ax = plt.subplots(
    figsize=(10, 6)
)

groups = [
    ("Classification", "Z-score"),
    ("Classification", "Depression map"),
    ("Classification", "NDVI"),
    ("Regression", "Z-score"),
    ("Regression", "Depression map"),
    ("Regression", "NDVI"),
]

positions = [1, 2, 3, 5, 6, 7]


# ---------------------------------------
# Prepare data
# ---------------------------------------
group_data = []

for model, ref in groups:

    subset = plot_df_fig1[
        (plot_df_fig1["model"] == model)
        & (plot_df_fig1["reference"] == ref)
    ].dropna(
        subset=["correlation"]
    )

    group_data.append(subset)


# ---------------------------------------
# Boxplots
# ---------------------------------------
ax.boxplot(
    [
        subset["correlation"]
        for subset in group_data
    ],
    positions=positions,
    widths=0.6,
    showfliers=False,
)


# ---------------------------------------
# Individual data points
# ---------------------------------------
for pos, subset, (model, ref) in zip(
    positions,
    group_data,
    groups,
):

    # Use model-specific threshold
    if model == "Classification":
        good = subset["cls_good"]
    else:
        good = subset["reg_good"]

    # Good prediction: gray circle
    ax.scatter(
        [pos] * good.sum(),
        subset.loc[
            good,
            "correlation",
        ],
        marker="o",
        color="gray",
        s=30,
        alpha=0.6,
        zorder=3,
        label=(
            "Above threshold"
            if pos == positions[0]
            else None
        ),
    )

    # Below threshold: red cross
    ax.scatter(
        [pos] * (~good).sum(),
        subset.loc[
            ~good,
            "correlation",
        ],
        marker="x",
        color="red",
        s=45,
        linewidths=1.5,
        alpha=0.9,
        zorder=4,
        label=(
            "Below threshold"
            if pos == positions[0]
            else None
        ),
    )


# ---------------------------------------
# Labels
# ---------------------------------------
ax.set_xticks(positions)

ax.set_xticklabels(
    [
        "Z-score",
        "Dep.",
        "NDVI",
        "Z-score",
        "Dep.",
        "NDVI",
    ],
    rotation=20,
)

ax.text(
    2,
    1.02,
    "Classification",
    ha="center",
    transform=ax.get_xaxis_transform(),
    fontsize=12,
)

ax.text(
    6,
    1.02,
    "Regression",
    ha="center",
    transform=ax.get_xaxis_transform(),
    fontsize=12,
)

ax.set_ylabel(
    "Spearman correlation"
)

ax.set_ylim(
    -1,
    1,
)

ax.grid(
    axis="y",
    alpha=0.3,
)

ax.legend()


# ---------------------------------------
# Layout
# ---------------------------------------
plt.tight_layout()


# ---------------------------------------
# Save
# ---------------------------------------
output_path = (
    DIR_EVAL /
    "fig_siamese-gradcam_correlation_boxplot.png"
)

fig.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight",
)

print(
    f"Saved: {output_path}"
)


# ---------------------------------------
# Display
# ---------------------------------------
plt.show()
plt.close(fig)

In [ ]:
# ---------------------------------------
# Figure 2 settings
# ---------------------------------------
references = [
    ("zscore", "Z-score"),
    ("depmap", "Depression map"),
    ("ndvi", "NDVI"),
]

CLS_THRESHOLD = 0.5
REG_THRESHOLD = 10.0


# ---------------------------------------
# Prepare plotting data
# ---------------------------------------
plot_data = eval_summary.copy()

# Classification quality
plot_data["cls_good"] = (
    plot_data["cls_prob"] >= CLS_THRESHOLD
)

# Regression quality
plot_data["reg_good"] = (
    plot_data["reg_pred"] >= REG_THRESHOLD
)


# ---------------------------------------
# Four prediction groups
# ---------------------------------------

# 1. Both good
plot_data["both_good"] = (
    plot_data["cls_good"]
    & plot_data["reg_good"]
)

# 2. Classification below threshold,
#    regression above threshold
plot_data["cls_not_good"] = (
    ~plot_data["cls_good"]
    & plot_data["reg_good"]
)

# 3. Regression below threshold,
#    classification above threshold
plot_data["reg_not_good"] = (
    plot_data["cls_good"]
    & ~plot_data["reg_good"]
)

# 4. Both below threshold
plot_data["both_not_good"] = (
    ~plot_data["cls_good"]
    & ~plot_data["reg_good"]
)


# ---------------------------------------
# Figure 2
# ---------------------------------------
fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5),
    sharex=True,
    sharey=True,
)


for ax, (ref, title) in zip(
    axes,
    references,
):

    x_col = f"corr_reg_{ref}"
    y_col = f"corr_cls_{ref}"

    # Remove patches for which either
    # correlation is undefined
    valid = (
        plot_data[x_col].notna()
        & plot_data[y_col].notna()
    )

    data = plot_data.loc[valid].copy()


    # -----------------------------------
    # 1. Both good
    # -----------------------------------
    mask = data["both_good"]

    ax.scatter(
        data.loc[mask, x_col],
        data.loc[mask, y_col],
        marker="o",
        color="gray",
        s=45,
        alpha=0.7,
        label="Both above threshold",
    )


    # -----------------------------------
    # 2. Classification below threshold
    # -----------------------------------
    mask = data["cls_not_good"]

    ax.scatter(
        data.loc[mask, x_col],
        data.loc[mask, y_col],
        marker="x",
        color="tab:red",
        s=55,
        linewidths=1.5,
        label="Classification below threshold",
    )


    # -----------------------------------
    # 3. Regression below threshold
    # -----------------------------------
    mask = data["reg_not_good"]

    ax.scatter(
        data.loc[mask, x_col],
        data.loc[mask, y_col],
        marker="^",
        color="tab:blue",
        s=55,
        alpha=0.8,
        label="Regression below threshold",
    )


    # -----------------------------------
    # 4. Both below threshold
    # -----------------------------------
    mask = data["both_not_good"]

    ax.scatter(
        data.loc[mask, x_col],
        data.loc[mask, y_col],
        marker="X",
        color="black",
        s=60,
        alpha=0.9,
        label="Both below threshold",
    )


    # -----------------------------------
    # 1:1 line
    # -----------------------------------
    ax.plot(
        [-1, 1],
        [-1, 1],
        "--",
        linewidth=1,
        color="gray",
    )


    # -----------------------------------
    # Patch IDs
    # -----------------------------------
    for _, row in data.iterrows():

        ax.text(
            row[x_col],
            row[y_col],
            row["patch_id"],
            fontsize=7,
        )


    # -----------------------------------
    # Panel settings
    # -----------------------------------
    ax.set_title(title)

    ax.set_xlim(-1, 1)
    ax.set_ylim(-1, 1)

    ax.grid(
        alpha=0.3,
    )


# ---------------------------------------
# Labels
# ---------------------------------------
axes[0].set_ylabel(
    "Classification Grad-CAM correlation"
)

for ax in axes:
    ax.set_xlabel(
        "Regression Grad-CAM correlation"
    )


# ---------------------------------------
# Legend
# ---------------------------------------
axes[0].legend(
    fontsize=9,
)


# ---------------------------------------
# Title
# ---------------------------------------
fig.suptitle(
    "Grad-CAM Correlation Comparison",
    fontsize=14,
)


# ---------------------------------------
# Layout
# ---------------------------------------
plt.tight_layout()


# ---------------------------------------
# Save
# ---------------------------------------
output_path = (
    DIR_EVAL /
    "fig_siamese-gradcam_correlation_comparison.png"
)

fig.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight",
)

print(
    f"Saved: {output_path}"
)


# ---------------------------------------
# Display
# ---------------------------------------
plt.show()
plt.close(fig)

In [ ]:
import rasterio
import matplotlib.pyplot as plt


def read_raster(path):
    with rasterio.open(path) as src:
        return src.read(1)


def show_patch(
    groups_patch_paths,
    eval_summary,
    patch_id,
    save=False,
    output_dir="."
):
    row = groups_patch_paths.loc[
        groups_patch_paths["patch_id"] == patch_id
    ].iloc[0]
    stats = eval_summary.loc[
        eval_summary["patch_id"] == patch_id
    ].iloc[0]

    sar_base = read_raster(row["sar_base"])
    sar_target = read_raster(row["sar_target"])
    zscore = read_raster(row["zscore"])
    depmap = read_raster(row["depmap"])
    ndvi = read_raster(row["ndvi"])
    gradcam_cls = read_raster(row["gradcam_cls"])
    gradcam_reg = read_raster(row["gradcam_reg"])

    fig, axes = plt.subplots(1, 7, figsize=(18, 3.5))

    # SAR Base
    axes[0].imshow(sar_base, cmap="gray")
    axes[0].set_title("SAR Base")
    axes[0].axis("off")

    # SAR Target
    axes[1].imshow(sar_target, cmap="gray")
    axes[1].set_title("SAR Target")
    axes[1].axis("off")

    # Z-score
    im = axes[2].imshow(zscore, cmap="gray")
    axes[2].set_title("Z-score")
    axes[2].axis("off")
    plt.colorbar(im, ax=axes[2], fraction=0.046)

    # Depression
    im = axes[3].imshow(depmap, cmap="gray")
    axes[3].set_title("Depression")
    axes[3].axis("off")
    plt.colorbar(im, ax=axes[3], fraction=0.046)

    # NDVI
    im = axes[4].imshow(ndvi, cmap="gray")
    axes[4].set_title("NDVI")
    axes[4].axis("off")
    plt.colorbar(im, ax=axes[4], fraction=0.046)

    # Classification Grad-CAM
    axes[5].imshow(sar_target, cmap="gray")
    im = axes[5].imshow(
        gradcam_cls,
        cmap="jet",
        alpha=0.5,
        vmin=0,
        vmax=1,
    )
    axes[5].set_title(
        f"Classification\nP={stats['cls_prob']:.3f}"
    )
    axes[5].axis("off")
    plt.colorbar(im, ax=axes[5], fraction=0.046)

    # Regression Grad-CAM
    axes[6].imshow(sar_target, cmap="gray")
    im = axes[6].imshow(
        gradcam_reg,
        cmap="jet",
        alpha=0.5,
        vmin=0,
        vmax=1,
    )
    axes[6].set_title(
        f"Regression\nPred={stats['reg_pred']:.3f}"
    )
    axes[6].axis("off")
    plt.colorbar(im, ax=axes[6], fraction=0.046)

    fig.suptitle(patch_id, fontsize=14)
    plt.tight_layout()
    if save:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

        output_file = output_dir / f"patches_{row['pair_id']}_{patch_id}.png"

        plt.savefig(
            output_file,
            dpi=300,
            bbox_inches="tight",
        )

        print(f"Saved to {output_file}")

    plt.show()
    plt.close(fig)

In [ ]:
patch_centers = pd.read_csv(FILEPATH_PATCH_CENTERS)
patches = patch_centers[patch_centers["usage"]=="test"]["id"].to_list()
patches

for patch in patches:
    show_patch(
        groups_patch_paths,
        eval_summary,
        patch,
        save=True,
        output_dir=DIR_EVAL / "patches"
    )